# Clustering Analysis — S2W8A2 Stage 2

WILDA Team 4 · Customer Churn Analysis

Produces the Clustering Analysis deliverables:

1. The optimal number of clusters, identified using the elbow method
2. A K-Means model trained on the dataset
3. Clusters visualised and labelled for interpretation and actionable insights

**Run `Data_Preparation/data_preparation.ipynb` first** — this notebook reads
the training set it produces.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42
K_RANGE = range(2, 11)

ROOT = Path.cwd().parent if Path.cwd().name == "Clustering_Analysis" else Path.cwd()
PREP = ROOT / "Data_Preparation"
OUT = ROOT / "Clustering_Analysis"
PLOTS = OUT / "cluster_visualisations"
PLOTS.mkdir(parents=True, exist_ok=True)

# Colour-blind safe palette, still readable printed in greyscale.
PALETTE = ["#4269d0", "#efb118", "#ff725c", "#6cc5b0", "#3ca951", "#a463f2"]
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 150, "font.size": 10,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25,
    "figure.facecolor": "white", "axes.facecolor": "white",
})

## 1. Load the prepared training set

In [ ]:
train_path = PREP / "train_set.csv"
if not train_path.exists():
    raise FileNotFoundError(
        f"{train_path} not found — run Data_Preparation/data_preparation.ipynb first."
    )

TARGET = "Churn"
train = pd.read_csv(train_path)
y_train = train[TARGET]
X_train = train.drop(columns=[TARGET])

print(f"Loaded {X_train.shape[0]:,} rows x {X_train.shape[1]} features (already scaled)")
X_train.head()

## 2. Choose the features to segment on

K-Means measures Euclidean distance, so a long tail of sparse one-hot dummy
columns can swamp the handful of features that actually describe customer
behaviour. Segmenting on the behavioural features gives tighter, more
interpretable clusters.

Set `CLUSTER_FEATURES` to the columns that describe behaviour in your dataset,
or leave it as `None` to use every feature and compare the two.

In [ ]:
CLUSTER_FEATURES = None  # e.g. ["tenure", "MonthlyCharges", "TotalCharges"]

X_seg = X_train if CLUSTER_FEATURES is None else X_train[CLUSTER_FEATURES]
print(f"Segmenting on {X_seg.shape[1]} features: {list(X_seg.columns)[:8]}"
      + (" ..." if X_seg.shape[1] > 8 else ""))

## 3. Elbow method — find the optimal number of clusters

Inertia (within-cluster sum of squares) always falls as k rises, so the elbow is
where the *rate* of improvement drops off. The silhouette score is plotted
alongside as a second opinion: higher is better, and it peaks at a genuine k
rather than falling forever.

In [ ]:
ks = list(K_RANGE)
inertia, silhouette = [], []

for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_seg)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(X_seg, km.labels_))
    print(f"k={k:2d}  inertia={km.inertia_:12,.0f}  silhouette={silhouette[-1]:.3f}")

metrics = pd.DataFrame({"k": ks, "inertia": inertia, "silhouette": silhouette})
metrics.to_csv(OUT / "cluster_selection_metrics.csv", index=False)

In [ ]:
best_i = int(np.argmax(silhouette))
OPTIMAL_K = ks[best_i]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

ax1.plot(ks, inertia, "o-", color=PALETTE[0], lw=2, ms=6)
ax1.axvline(OPTIMAL_K, color=PALETTE[2], ls="--", lw=1.5)
ax1.annotate(f"elbow at k={OPTIMAL_K}", xy=(OPTIMAL_K, inertia[best_i]),
             xytext=(14, 30), textcoords="offset points",
             color=PALETTE[2], fontweight="bold",
             arrowprops=dict(arrowstyle="->", color=PALETTE[2], lw=1.4))
ax1.set_title("Elbow method")
ax1.set_xlabel("Number of clusters (k)")
ax1.set_ylabel("Within-cluster sum of squares (inertia)")

ax2.bar([str(k) for k in ks], silhouette,
        color=[PALETTE[2] if k == OPTIMAL_K else "#c9d2e3" for k in ks])
ax2.set_title("Silhouette score by k")
ax2.set_xlabel("Number of clusters (k)")
ax2.set_ylabel("Mean silhouette score")
ax2.text(best_i, silhouette[best_i] + 0.004, f"{silhouette[best_i]:.3f}",
         ha="center", fontweight="bold", color=PALETTE[2])

fig.suptitle("Choosing the optimal number of clusters", y=1.03, fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(PLOTS / "elbow_method.png", bbox_inches="tight")
plt.show()

print(f"Optimal k = {OPTIMAL_K} (silhouette {silhouette[best_i]:.3f})")

Check the elbow plot before moving on. If the visual elbow disagrees with the
silhouette peak, set `OPTIMAL_K` by hand and say in the video why you chose it —
a reasoned override is fine, an unexamined default is not.

## 4. Train the K-Means model

In [ ]:
kmeans = KMeans(n_clusters=OPTIMAL_K, n_init=10, random_state=RANDOM_STATE).fit(X_seg)
labels = kmeans.labels_

joblib.dump(kmeans, OUT / "kmeans_model.pkl")
print(f"Trained K-Means with k={OPTIMAL_K}, saved to kmeans_model.pkl")
print(f"Cluster sizes: {pd.Series(labels).value_counts().sort_index().to_dict()}")

## 5. Profile the clusters

The labels have to come from the numbers, not from guesswork. This table is the
evidence behind every name assigned in the next cell.

In [ ]:
profile = X_seg.assign(Cluster=labels).groupby("Cluster").mean()
profile.insert(0, "size", pd.Series(labels).value_counts().sort_index().values)
profile["churn_rate"] = y_train.groupby(labels).mean().values

profile.round(3)

In [ ]:
# Name each cluster from its own profile. Adjust the rules to fit what the
# table above actually shows — these thresholds are a starting point.
overall_churn = y_train.mean()
names = {}

for c, row in profile.iterrows():
    if row.churn_rate > overall_churn * 1.4:
        names[c] = "At-risk churners"
    elif row.churn_rate < overall_churn * 0.6:
        names[c] = "Loyal customers"
    else:
        names[c] = f"Segment {c}"

# Two clusters landing on the same name means the rules above aren't yet
# separating them. Disambiguate by size so the legend stays readable, then go
# back and refine the rules using whatever actually distinguishes the two.
from collections import Counter

counts = Counter(names.values())  # snapshot: renaming as we go would skew it
seen = {}
for c in list(names):
    base = names[c]
    if counts[base] > 1:
        seen[base] = seen.get(base, 0) + 1
        names[c] = f"{base} {seen[base]}"

profile["label"] = [names[c] for c in profile.index]
profile.to_csv(OUT / "cluster_profiles.csv")

for c in profile.index:
    print(f"Cluster {c}: {names[c]:22s} n={profile['size'][c]:5,}  "
          f"churn={profile['churn_rate'][c]:.1%}")

## 6. Visualise the clusters

PCA projects the data to two dimensions for plotting only — the model itself
uses all the segmentation features.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
points = pca.fit_transform(X_seg)
centres = pca.transform(kmeans.cluster_centers_)
var = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(8.5, 6))
for c in range(OPTIMAL_K):
    m = labels == c
    ax.scatter(points[m, 0], points[m, 1], s=22, alpha=0.6,
               color=PALETTE[c % len(PALETTE)],
               label=f"{names[c]}  (n={int(m.sum()):,}, churn {profile['churn_rate'][c]:.0%})")

ax.scatter(centres[:, 0], centres[:, 1], marker="X", s=260,
           c="white", edgecolors="black", linewidths=1.8, zorder=5)
for c, (px, py) in enumerate(centres):
    ax.annotate(names[c], (px, py), xytext=(0, 18), textcoords="offset points",
                ha="center", fontweight="bold", fontsize=9, zorder=6,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#00000022"))

ax.set_title(f"Customer segments from K-Means (k={OPTIMAL_K})")
ax.set_xlabel(f"Principal component 1 ({var[0]:.0%} of variance)")
ax.set_ylabel(f"Principal component 2 ({var[1]:.0%} of variance)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), frameon=False)
fig.tight_layout()
fig.savefig(PLOTS / "cluster_visualisation.png", bbox_inches="tight")
plt.show()

In [ ]:
# Profile heatmap — shows at a glance what makes each segment different.
cols = [c for c in profile.columns if c not in ("size", "label")]
z = profile[cols].apply(lambda s: (s - s.mean()) / (s.std() or 1))

fig, ax = plt.subplots(figsize=(1.1 * len(cols) + 3, 0.95 * OPTIMAL_K + 2.4))
im = ax.imshow(z.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(cols)), cols, rotation=45, ha="right")
ax.set_yticks(range(OPTIMAL_K),
              [f"{names[c]}\n(n={profile['size'][c]:,})" for c in profile.index])
ax.grid(False)
for i in range(OPTIMAL_K):
    for j, col in enumerate(cols):
        v = profile[col].iloc[i]
        txt = f"{v:.0%}" if col == "churn_rate" else f"{v:,.2f}"
        ax.text(j, i, txt, ha="center", va="center", fontsize=8, fontweight="bold",
                color="white" if abs(z.values[i, j]) > 1.1 else "#1a1a1a")

ax.set_title("Cluster profiles  (colour = relative to the other clusters)")
fig.colorbar(im, ax=ax, shrink=0.7, label="standard deviations from mean")
fig.tight_layout()
fig.savefig(PLOTS / "cluster_profiles.png", bbox_inches="tight")
plt.show()

## 7. Findings and actionable insights

Replace the placeholders below with what the profile table actually shows. This
is the "actionable insights" half of the deliverable, and it is what the video
should spend most of its time on.

| Segment | Size | Churn rate | What characterises it | Recommended action |
|---|---|---|---|---|
| … | … | … | … | … |

In [ ]:
# Cluster assignments, for the Stage 3 predictive model to use as a feature.
pd.DataFrame({
    "row_index": X_seg.index,
    "cluster": labels,
    "cluster_label": [names[c] for c in labels],
}).to_csv(OUT / "cluster_assignments.csv", index=False)

print("Saved cluster_assignments.csv")
print(f"\nDeliverables written to:\n  {OUT}\n  {PLOTS}")